# Vector scaling / Dirichlet-style matrix scaling on the 0.915 model (no training)

Runtime > any GPU (L4/T4 fine), then Run all. A few minutes.

Unlike temperature scaling, these maps can change which class wins: vector scaling (Guo et al. 2017) learns a weight and bias per class; Dirichlet calibration with ODIR regularisation (Kull et al. 2019) learns a regularised 10x10 matrix on log-probabilities. Fitted on **val** only, judged by 2-fold cross-fitting on val; internal test printed as the honest check. A new `submission.csv` is written only if the cross-fitted val accuracy beats the current setting by more than 0.2 pt (5 of 2,500). The hidden test images are only predicted, never used to fit anything.


## 1. Paths, Drive, device

In [1]:
# ================= Vector scaling / Dirichlet-style matrix scaling on the 0.915 model (no training) =================
import os, re, glob, time, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- paths: set RUN_NAME if you know it; otherwise the run whose submission has internal test 0.9812 is found automatically
RUN_NAME = None                       # e.g. "kd_sewrn_only_student_..." (the 0.915 run)
FIND_TEST_TAG = "test0.9812"          # used only when RUN_NAME is None
try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive; drive.mount("/content/drive")
    ROOT, WORK_DIR = "/content/drive/MyDrive/cifar10_resnet_challenge", "/content/work"
except ImportError:
    IN_COLAB = False
    ROOT = os.environ.get("TTA_ROOT", "/scratch/ds8223/cifar10_resnet_challenge"); WORK_DIR = os.environ.get("TTA_WORK", "/scratch/ds8223/kd_work")
CACHE_DIR, SPLIT_DIR = os.path.join(ROOT, "data", "cache"), os.path.join(ROOT, "data", "splits")

if RUN_NAME is None:
    hits = sorted({os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(ROOT, "runs", "*", f"submission_*{FIND_TEST_TAG}*.csv"))})
    print("runs with a submission tagged", FIND_TEST_TAG, ":", hits)
    assert len(hits) == 1, "found 0 or several runs: set RUN_NAME by hand (see the list above)"
    RUN_NAME = hits[0]
RUN_DIR = os.path.join(ROOT, "runs", RUN_NAME)
BEST_CKPT = os.path.join(RUN_DIR, "best.pt")
print("run:", RUN_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (slow: use a GPU)")
AMP_DTYPE = torch.bfloat16 if device.type == "cpu" or torch.cuda.is_bf16_supported() else None

Mounted at /content/drive
runs with a submission tagged test0.9812 : ['kd_sewrn_only_student_resnext29_slasd_400ep_v1']
run: kd_sewrn_only_student_resnext29_slasd_400ep_v1
device: NVIDIA A100-SXM4-40GB


## 2. Model code

In [2]:
# ---- Models (same code as the training notebooks): ResNet, ResNeXt-29, Shake-Shake, chosen with CFG["ARCH"] ----
class BasicBlock(nn.Module):
    """Post-activation basic block (our original ResNet)."""
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class CifarResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256), blocks=(3, 3, 3), num_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, cin = [], widths[0]
        for i, (w, nb) in enumerate(zip(widths, blocks)):
            for j in range(nb):
                layers.append(BasicBlock(cin, w, 2 if (j == 0 and i > 0) else 1)); cin = w
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn2.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class ResNeXtBlock(nn.Module):
    """ResNeXt bottleneck: 1x1 conv -> grouped 3x3 conv (cardinality groups) -> 1x1 conv, plus skip (Xie et al. 2017)."""
    def __init__(self, cin, inner, cout, stride, cardinality):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, inner, 1, bias=False); self.bn1 = nn.BatchNorm2d(inner)
        self.conv2 = nn.Conv2d(inner, inner, 3, stride, 1, groups=cardinality, bias=False); self.bn2 = nn.BatchNorm2d(inner)
        self.conv3 = nn.Conv2d(inner, cout, 1, bias=False); self.bn3 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Identity()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class ResNeXt29(nn.Module):
    """CIFAR ResNeXt-29: 3x3 stem (64) + 3 stages x 3 blocks; output widths 256/512/1024;
    inner width = cardinality * bottleneck_width, doubled at each stage. (2, 64) is the standard 2x64d."""
    def __init__(self, cardinality=2, bottleneck_width=64, blocks=(3, 3, 3), out_widths=(256, 512, 1024),
                 num_classes=10, dropout=0.0, stem=64, strides=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, stem, 3, 1, 1, bias=False), nn.BatchNorm2d(stem), nn.ReLU(inplace=True))
        layers, cin = [], stem
        for i, (cout, nb) in enumerate(zip(out_widths, blocks)):
            inner = cardinality * bottleneck_width * (2 ** i)
            for j in range(nb):
                layers.append(ResNeXtBlock(cin, inner, cout, strides[i] if j == 0 else 1, cardinality)); cin = cout
        self.layers = nn.Sequential(*layers)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        for b in self.layers: nn.init.zeros_(b.bn3.weight)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(self.layers(self.stem(x)), 1), 1)

class SE(nn.Module):
    """Squeeze-excitation (Hu et al. 2018): channel weights from global average pooling. Teachers only."""
    def __init__(self, c, r=16):
        super().__init__()
        self.fc1, self.fc2 = nn.Conv2d(c, c // r, 1), nn.Conv2d(c // r, c, 1)
    def forward(self, x):
        return x * torch.sigmoid(self.fc2(F.relu(self.fc1(F.adaptive_avg_pool2d(x, 1)))))

class WBlock(nn.Module):
    """Pre-activation wide residual block (Zagoruyko & Komodakis 2016)."""
    def __init__(self, cin, cout, stride, drop, se_r=0):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(cin), nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(cout), nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.dropout = nn.Dropout(drop)
        self.skip = None if (stride == 1 and cin == cout) else nn.Conv2d(cin, cout, 1, stride, bias=False)
        if se_r: self.se = SE(cout, se_r)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.skip is None else self.skip(o)
        o = self.conv2(self.dropout(F.relu(self.bn2(self.conv1(o)))))
        if hasattr(self, "se"): o = self.se(o)
        return o + sc

class WRN(nn.Module):
    """WideResNet for CIFAR (Zagoruyko & Komodakis 2016). k = widen factor (WRN-28-10: 36.5M params) or explicit stage widths."""
    def __init__(self, depth=28, k=10, dropout=0.0, num_classes=10, se_r=0):
        super().__init__()
        n = (depth - 4) // 6
        w = [16] + (list(k) if isinstance(k, (tuple, list)) else [16 * k, 32 * k, 64 * k])   # k: widen factor, or the 3 stage widths
        self.stem = nn.Conv2d(3, w[0], 3, 1, 1, bias=False)
        layers, cin = [], w[0]
        for i in range(3):
            for j in range(n):
                layers.append(WBlock(cin, w[i + 1], 2 if (j == 0 and i > 0) else 1, dropout, se_r)); cin = w[i + 1]
        self.layers = nn.Sequential(*layers); self.bn = nn.BatchNorm2d(cin)
        self.drop = nn.Dropout(0.0); self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def features(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.stem(x)))), 1), 1)
    def forward(self, x):
        return self.fc(self.drop(self.features(x)))

def _build_base(c):
    """Build any of our architectures from a config dict (also rebuilds teachers from their saved cfg)."""
    arch = c.get("ARCH", "resnet")
    if arch == "resnet":
        return CifarResNet(c["WIDTHS"], c["BLOCKS"], dropout=c.get("DROPOUT", 0.0))
    if arch == "resnext":
        return ResNeXt29(c.get("CARDINALITY", 2), c.get("BOTTLENECK_WIDTH", 64), c.get("BLOCKS", (3, 3, 3)),
                         c.get("WIDTHS", (256, 512, 1024)), dropout=c.get("DROPOUT", 0.0),
                         strides=tuple(c.get("STRIDES", (1, 2, 2))))   # (1, 1, 2) = higher-resolution student, same params
    if arch in ("wrn", "sewrn"):     # sewrn = WRN with SE blocks: a training-only teacher
        return WRN(c["DEPTH"], c["WIDEN"], c.get("DROPOUT", 0.0), se_r=c.get("SE_REDUCTION", 16) if arch == "sewrn" else 0)
    raise ValueError(f"unknown ARCH {arch}")

def build_model(c):
    """Base network; with SLA on, adds the joint (class x rotation) head used only during training."""
    net = _build_base(c)
    if c.get("SLA", False):
        net.fc_joint = nn.Linear(net.fc.in_features, 10 * c.get("SLA_ROTATIONS", 4))
    return net

def forward_both(net, x):
    """Returns (10-class self-distillation head logits, joint class x rotation head logits)."""
    f = net.drop(net.features(x))
    return net.fc(f), net.fc_joint(f)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Load the 0.915 best.pt, data and split

In [3]:
# ---- checkpoint, data, split (same split file as every run)
best = torch.load(BEST_CKPT, map_location=device, weights_only=False)
CFG = best["cfg"]
X_full = np.load(os.path.join(CACHE_DIR, "train_x.npy")); y_full = np.load(os.path.join(CACHE_DIR, "train_y.npy"))
X_test = np.load(os.path.join(CACHE_DIR, "test_x.npy")); test_ids = np.load(os.path.join(CACHE_DIR, "test_ids.npy"))
SAMPLE_SUB = pd.read_csv(os.path.join(CACHE_DIR, "sample_submission.csv"))
sp = np.load(os.path.join(SPLIT_DIR, f"split_seed{CFG['SPLIT_SEED']}_val{CFG['VAL_FRAC']}_test{CFG['TEST_FRAC']}_n{len(y_full)}.npz"))
va_idx, te_idx = sp["val"], sp["test"]
def to_dev(x): return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(device)
Xva, yva = to_dev(X_full[va_idx]), torch.from_numpy(y_full[va_idx]).to(device)
Xte_int, yte_int = to_dev(X_full[te_idx]), torch.from_numpy(y_full[te_idx]).to(device)
Xkaggle = to_dev(X_test)
MEAN, STD = best["norm"]["mean"].to(device), best["norm"]["std"].to(device)   # exact normalisation of that run
def normalize(x_u8): return ((x_u8.float() / 255 - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def _ag_logits(m, xb):
    """SLA+AG: average the joint-head logit of (class, k) on the image rotated by k*90 degrees."""
    agg = 0
    for k in range(CFG.get("SLA_ROTATIONS", 4)):
        _, j = forward_both(m, torch.rot90(xb, k, dims=(2, 3)))
        agg = agg + j.float().view(len(xb), 10, -1)[:, :, k]
    return agg / CFG.get("SLA_ROTATIONS", 4)

net = build_model(CFG).to(device).to(memory_format=torch.channels_last)
net.load_state_dict(best["model"]); net.eval()
has_joint = getattr(net, "fc_joint", None) is not None
print(f"params {count_params(net):,} | saved val {best['val_acc']:.4f} | SLA head: {has_joint} | val {len(Xva)} | internal test {len(Xte_int)} | kaggle {len(Xkaggle)}")

params 4,998,738 | saved val 0.9856 | SLA head: True | val 2500 | internal test 2500 | kaggle 2000


## 4. Fit on val, judge by cross-fitting

In [4]:
# ---- logits of the 0.915 model (10-class head), averaged over image + mirror ----
@torch.no_grad()
def flip_logits(X, bs=500):
    out = []
    for i in range(0, len(X), bs):
        xb = normalize(X[i:i + bs])
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            z = 0.5 * (net(xb).float() + net(xb.flip(3)).float())
            p = net(xb).float().softmax(1) + net(xb.flip(3)).float().softmax(1)   # exactly how the 0.915 submission scored
        out.append((z, p))
    return torch.cat([o[0] for o in out]), torch.cat([o[1] for o in out])
t0 = time.time()
Zva, Pva = flip_logits(Xva); Zte, Pte = flip_logits(Xte_int); Zkg, Pkg = flip_logits(Xkaggle)
acc = lambda s, y: (s.argmax(1) == y).float().mean().item()
print(f"logits computed in {time.time() - t0:.0f}s")
print(f"CURRENT (0.915 submission, sum of softmax): val {acc(Pva, yva):.4f} | internal test {acc(Pte, yte_int):.4f}   (expected 0.9852 / 0.9812)")
print(f"same model, mean logits:                    val {acc(Zva, yva):.4f} | internal test {acc(Zte, yte_int):.4f}")

# ---- calibration maps, fitted by minimising cross-entropy on VAL images only ----
# vector scaling (Guo et al. 2017): z' = w * z + b  (per-class weight and bias, 20 numbers)
# Dirichlet calibration with ODIR (Kull et al. 2019): z' = W log_softmax(z) + b, L2 penalty on off-diagonal W and on b
def fit_map(Z, y, kind, lam=0.0, steps=500):
    K = Z.shape[1]
    if kind == "vector":
        w = torch.ones(K, device=Z.device, requires_grad=True); b = torch.zeros(K, device=Z.device, requires_grad=True); params = [w, b]
        f = lambda Zx: Zx * w + b
    else:
        W = torch.eye(K, device=Z.device).requires_grad_(); b = torch.zeros(K, device=Z.device, requires_grad=True); params = [W, b]
        off = 1 - torch.eye(K, device=Z.device)
        f = lambda Zx: F.log_softmax(Zx, 1) @ W.T + b
    opt = torch.optim.LBFGS(params, lr=0.5, max_iter=steps, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(f(Z), y)
        if kind == "dirichlet": loss = loss + lam * (((W * off) ** 2).sum() / (K * (K - 1)) + (b ** 2).mean())
        loss.backward(); return loss
    opt.step(closure)
    with torch.no_grad(): return lambda Zx: f(Zx).detach()

# 2-fold CROSS-FITTING on val: fit on one half, score the other half (so the val number is honest without the internal test)
g = torch.Generator().manual_seed(0); idx = torch.randperm(len(yva), generator=g).to(device); h = len(idx) // 2
folds = [(idx[:h], idx[h:]), (idx[h:], idx[:h])]
def cv(kind, lam=0.0):
    correct, nll = 0, 0.0
    for a, b in folds:
        m = fit_map(Zva[a], yva[a], kind, lam)
        out = m(Zva[b]); correct += (out.argmax(1) == yva[b]).sum().item(); nll += F.cross_entropy(out, yva[b], reduction="sum").item()
    return correct / len(yva), nll / len(yva)

rows = [{"method": "vector scaling", "lam": None, **dict(zip(("cv_val_acc", "cv_val_nll"), cv("vector")))}]
for lam in (1e-3, 1e-2, 1e-1, 1.0):
    rows.append({"method": "dirichlet (ODIR)", "lam": lam, **dict(zip(("cv_val_acc", "cv_val_nll"), cv("dirichlet", lam)))})
res = pd.DataFrame(rows)
print("\n2-fold cross-fitted on VAL (accuracy and NLL on the held-out half of val):")
print(res.round(4).to_string(index=False))

# pick the regularisation by cross-fitted NLL (not accuracy), then judge it by cross-fitted accuracy
d = res[res.method == "dirichlet (ODIR)"].sort_values("cv_val_nll").iloc[0]
cands = [res.iloc[0], d]
best = max(cands, key=lambda r: r.cv_val_acc)
cur_val = acc(Pva, yva)
use = best.cv_val_acc > cur_val + 0.002
kind = "vector" if best.method == "vector scaling" else "dirichlet"
m_all = fit_map(Zva, yva, kind, 0.0 if best.lam is None or pd.isna(best.lam) else float(best.lam))
te_acc = acc(m_all(Zte), yte_int)
print(f"\nCURRENT: val {cur_val:.4f} | internal test {acc(Pte, yte_int):.4f}")
print(f"BEST MAP ({best.method}{'' if best.lam is None or pd.isna(best.lam) else f', lam {best.lam}'}): cross-fitted val {best.cv_val_acc:.4f} | "
      f"internal test (fit on all val) {te_acc:.4f} -> change {100 * (te_acc - acc(Pte, yte_int)):+.2f} pt ({int(round((te_acc - acc(Pte, yte_int)) * len(yte_int))):+d} images)")
print("DECISION:", "use it" if use else "keep the 0.915 submission (no cross-fitted VAL gain > 0.2 pt)")

if use:
    kp = m_all(Zkg).argmax(1).cpu().numpy(); kb = Pkg.argmax(1).cpu().numpy()
    sub = pd.DataFrame({"id": test_ids.astype(int), "label": kp.astype(int)})
    assert len(sub) == len(SAMPLE_SUB) and (sub["id"].values == SAMPLE_SUB["id"].values).all() and sub["label"].between(0, 9).all()
    drive_sub = os.path.join(RUN_DIR, f"submission_calib_{kind}_cvval{best.cv_val_acc:.4f}_test{te_acc:.4f}.csv"); sub.to_csv(drive_sub, index=False)
    os.makedirs(os.path.join(WORK_DIR, "calib"), exist_ok=True); local_sub = os.path.join(WORK_DIR, "calib", "submission.csv"); sub.to_csv(local_sub, index=False)
    print(f"saved {drive_sub}\nKaggle predictions that differ from the 0.915 submission: {(kp != kb).sum()} of {len(kp)}")
    if IN_COLAB:
        from google.colab import files; files.download(local_sub)

logits computed in 5s
CURRENT (0.915 submission, sum of softmax): val 0.9852 | internal test 0.9812   (expected 0.9852 / 0.9812)
same model, mean logits:                    val 0.9852 | internal test 0.9812

2-fold cross-fitted on VAL (accuracy and NLL on the held-out half of val):
          method   lam  cv_val_acc  cv_val_nll
  vector scaling   NaN      0.9844      0.2940
dirichlet (ODIR) 0.001      0.9796      0.4387
dirichlet (ODIR) 0.010      0.9828      0.2426
dirichlet (ODIR) 0.100      0.9848      0.1386
dirichlet (ODIR) 1.000      0.9840      0.0960

CURRENT: val 0.9852 | internal test 0.9812
BEST MAP (vector scaling): cross-fitted val 0.9844 | internal test (fit on all val) 0.9800 -> change -0.12 pt (-3 images)
DECISION: keep the 0.915 submission (no cross-fitted VAL gain > 0.2 pt)
